In [2]:
!pip install google-genai
from google.genai import Client
import re

# 설정
client = Client(api_key="-")
MODEL_NAME = "gemma-3-1b-it"

def extract_num(text):
    numbers = re.findall(r"[-+]?\d*\.\d+|\d+", text.replace(',', ''))
    return numbers[-1] if numbers else None

In [4]:
# GSM8K 데이터셋에서 문제 선택
from datasets import load_dataset
dataset = load_dataset("gsm8k", "main", split='test')
ex = dataset[0]
question = ex['question']
gt = extract_num(ex['answer'])

print(f"문제: {question}")
print(f"정답: {gt}\n" + "-"*30)

문제: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?
정답: 18
------------------------------


In [5]:
import time

# Standard
std_res = client.models.generate_content(model=MODEL_NAME, contents=f"Q: {question}\n추가적인 설명 없이, 정답만 말해줘.\n답은?").text
print(f"결과: {std_res}")
time.sleep(60)

# CoT
cot_res = client.models.generate_content(model=MODEL_NAME, contents=f"Q: {question}\nStep by step으로 사고과정과 정답을 말해줘.\n답은?.").text
print(f"결과: {cot_res}")
time.sleep(60)

# Self-Refine
refine_res = client.models.generate_content(model=MODEL_NAME, contents=f"Q: {question}\n이전의 대답: {cot_res}\n이전의 대답에서 오류를 찾고, 고쳐서 최종 답을 말해줘.\n답은?").text
print(f"결과: {refine_res}")

결과: $14
결과: Okay, let's break this problem down step-by-step to find the answer.

**1. Calculate the number of eggs remaining after breakfast:**

* Janet lays 16 eggs per day.
* She eats 3 eggs, so she has 16 - 3 = 13 eggs left.

**2. Calculate the number of eggs sold:**

* She sells each duck egg for $2.
* She has 13 eggs, so she sells 13 * $2 = $26.

**3. Calculate the total daily earnings:**

* She makes $26 selling the eggs.

**Answer:** Janet makes $26 every day at the farmers' market.

**Final Answer: $26**
결과: 이전의 답변에서 오류가 있었습니다. 

* **계산 오류:** 13 eggs remaining는 13 * $2 = $26이 아니라, 13 * $2 = $26입니다.
* **계산 오류:** 13 eggs remaining는 13 * $2 = $26이 아니라, 13 * $2 = $26입니다.

정확한 계산은 다음과 같습니다.

1. **Calculate the number of eggs remaining after breakfast:** Janet lays 16 eggs per day. She eats 3 eggs for breakfast, so she has 16 - 3 = 13 eggs left.
2. **Calculate the number of eggs sold:** She sells each duck egg for $2. She has 13 eggs, so she sells 13 * $2 = $26.
3. **Calculate the t